# Titanic - Kaggle Notebook

شغل نفس الـ pipeline بتاع المشروع بس جوه Jupyter عشان يترفع على Kaggle.
بيشتغل في الحالتين: ملفات كاجل (`train.csv` / `test.csv`) أو ملفات seaborn المحلية (`titanic.csv`).

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc

In [ ]:
from pathlib import Path

# Kaggle paths first, local fallback second
if Path('/kaggle/input/titanic/train.csv').exists():
    train = pd.read_csv('/kaggle/input/titanic/train.csv')
    test = pd.read_csv('/kaggle/input/titanic/test.csv')
    IS_KAGGLE = True
elif Path('train.csv').exists():
    train = pd.read_csv('train.csv')
    test = pd.read_csv('test.csv')
    IS_KAGGLE = True
else:
    train = pd.read_csv('titanic.csv')  # نسخة seaborn المحلية
    test = None
    IS_KAGGLE = False

print('IS_KAGGLE:', IS_KAGGLE)
print(train.shape)
print(list(train.columns))
train.head()

In [ ]:
# استكشاف سريع
print(train.info())
print(train.isnull().sum())
print(train['Survived'].mean() if 'Survived' in train else train['survived'].mean())

In [ ]:
def extract_title(name):
    m = re.search(r',\s*(\w+)\.', str(name))
    return m.group(1) if m else 'Unknown'

def clean(df, is_train=True, age_median=None, fare_median=None):
    df = df.copy()
    y_col = 'Survived' if 'Survived' in df.columns else 'survived'

    # Title من Name (كاجل بس)
    if 'Name' in df.columns:
        df['Title'] = df['Name'].apply(extract_title).replace({
            'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
            'Capt': 'Officer', 'Col': 'Officer', 'Major': 'Officer', 'Dr': 'Officer', 'Rev': 'Officer',
            'Don': 'Royalty', 'Sir': 'Royalty', 'Lady': 'Royalty', 'Countess': 'Royalty', 'Jonkheer': 'Royalty', 'Dona': 'Royalty'
        })
    # Deck من Cabin (كاجل بس)
    if 'Cabin' in df.columns:
        df['Deck'] = df['Cabin'].str[0].fillna('M')
    if 'deck' in df.columns and 'Deck' not in df.columns:
        df = df.drop(columns=['deck'])  # 77% ناقص في نسخة seaborn

    # Age / Fare imputation
    if age_median is None:
        age_median = df['Age'].median() if 'Age' in df.columns else df['age'].median()
    if fare_median is None:
        fcol = 'Fare' if 'Fare' in df.columns else 'fare'
        fare_median = df[fcol].median()
    for c in ['Age', 'age']:
        if c in df.columns:
            df[c] = df[c].fillna(age_median)
    for c in ['Fare', 'fare']:
        if c in df.columns:
            df[c] = df[c].fillna(fare_median)
    for c in ['Embarked', 'embarked']:
        if c in df.columns and df[c].isnull().any():
            df[c] = df[c].fillna(df[c].mode()[0])

    # توحيد أسماء الأعمدة لنسخة كاجل
    rename = {'Survived': 'survived', 'Pclass': 'pclass', 'Sex': 'sex', 'Age': 'age',
              'SibSp': 'sibsp', 'Parch': 'parch', 'Fare': 'fare', 'Embarked': 'embarked'}
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    # features
    df['family_size'] = df['sibsp'] + df['parch'] + 1
    df['is_child'] = (df['age'] < 12).astype(int)
    df['fare_per_person'] = df['fare'] / df['family_size']
    df['fare_log'] = np.log1p(df['fare'])
    df['age_x_pclass'] = df['age'] * df['pclass']
    return df, age_median, fare_median

train_c, age_med, fare_med = clean(train)
test_c = clean(test, is_train=False, age_median=age_med, fare_median=fare_med)[0] if test is not None else None
print(train_c.shape)
train_c.head()

In [ ]:
# تحليل بصري
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
train_c.groupby('sex')['survived'].mean().plot(kind='bar', ax=axes[0], title='Survival by sex')
train_c.groupby('pclass')['survived'].mean().plot(kind='bar', ax=axes[1], title='Survival by pclass')
plt.tight_layout()
plt.show()

if 'Title' in train_c.columns:
    print(train_c.groupby('Title')['survived'].mean().sort_values(ascending=False))

In [ ]:
# تدريب baseline
base_features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked',
                   'family_size', 'is_child', 'fare_per_person', 'fare_log', 'age_x_pclass']
if 'Title' in train_c.columns:
    base_features += ['Title']
if 'Deck' in train_c.columns:
    base_features += ['Deck']
if 'who' in train_c.columns:  # نسخة seaborn
    base_features += ['who']

num_cols = [c for c in base_features if train_c[c].dtype != object]
cat_cols = [c for c in base_features if train_c[c].dtype == object]
print('features:', base_features)

X = train_c[base_features]
y = train_c['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

for name, clf in [
    ('LR', LogisticRegression(max_iter=1000, C=0.1)),
    ('RF', RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)),
    ('HGB', HistGradientBoostingClassifier(random_state=42)),
]:
    pipe = Pipeline([('prep', preprocess), ('clf', clf)])
    pipe.fit(X_train, y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    cv = cross_val_score(pipe, X, y, cv=5).mean()
    print(f'{name}: test={acc:.4f} cv={cv:.4f}')

In [ ]:
# Tuning لأحسن موديل + حفظه
grid = GridSearchCV(
    Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=1000))]),
    {'clf__C': [0.05, 0.1, 0.5, 1.0]}, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)
print('best:', grid.best_params_, f"cv={grid.best_score_:.4f}",
      f"test={accuracy_score(y_test, grid.predict(X_test)):.4f}")

best_model = grid.best_estimator_

# ROC
proba = best_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, proba)
print('AUC:', auc(fpr, tpr))
plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.3f}")
plt.plot([0, 1], [0, 1], '--')
plt.legend()
plt.show()

In [ ]:
# submission لكاجل (لو test.csv موجود) وإلا أمثلة
if test_c is not None:
    preds = best_model.predict(test_c[base_features])
    pid = test_c['PassengerId'] if 'PassengerId' in test_c.columns else test_c.index
    sub = pd.DataFrame({'PassengerId': pid, 'Survived': preds})
    sub.to_csv('submission.csv', index=False)
    print(sub.shape)
    sub.head()
else:
    print('مفيش test.csv — جرب راكب جديد:')
    ex = pd.DataFrame([{c: X.iloc[0][c] for c in base_features}])
    print(best_model.predict(ex), best_model.predict_proba(ex))